# ⚛️ ENTRAÎNEMENT DU CERVEAU CONVERSATIONNEL RATISS-ONE SUR GOOGLE COLAB
**RATISS Labs · Auteur : Jonathan Evina (Yaoundé, Cameroun) · Licence MIT**  
**Architecture : Réseau de Neurones Intriqués (RNI) · Tissage Transversal Q ➔ R**  
**Dataset Réel : CATIE-AQ NewsQuadFR (69 300 Paires Q/R françaises, Parquet ungated)**  

Ce notebook télécharge automatiquement le Cerveau Maître de **6 Mo** depuis `RATISS-FIRST`, télécharge le gros dataset conversationnel officiel de **69 300 paires de questions-réponses**, tisse les liaisons synaptiques stimulus-réponse à **~600 000 mots par seconde**, et sauvegarde le cerveau augmenté scellé SHA-256 directement sur votre **Google Drive** !

### 🔗 1/6 : Connexion à Google Drive

In [ ]:
from google.colab import drive
import os, sys, time, json, gzip, hashlib, re, urllib.request, io
import pandas as pd

print("🔗 1/6 : Connexion à Google Drive...")
drive.mount('/content/drive')
dossier_drive = '/content/drive/MyDrive/RATISS_CERVEAUX'
os.makedirs(dossier_drive, exist_ok=True)
print(f"✅ Dossier Google Drive prêt : {dossier_drive}")

### 📦 2/6 : Téléchargement du Cerveau Maître RNI depuis RATISS-FIRST

In [ ]:
print("📦 2/6 : Téléchargement du Cerveau Maître depuis RATISS-FIRST (GitHub)...")
url_modele = "https://raw.githubusercontent.com/jonathansearch/RATISS-FIRST/main/RatissOne.ratiss"
chemin_local = "/content/RatissOne_Initial.ratiss"

urllib.request.urlretrieve(url_modele, chemin_local)
taille_mo = os.path.getsize(chemin_local) / (1024 * 1024)
print(f"✅ Cerveau téléchargé : {chemin_local} ({taille_mo:.2f} Mo)")

t0 = time.time()
with gzip.open(chemin_local, "rt", encoding="utf-8") as f:
    cerveau = json.load(f)

neurones = cerveau["RNI_MATRICE"]["neurones"]
liens = cerveau["RNI_MATRICE"]["liens"]
print(f"🧠 Cerveau chargé en {time.time()-t0:.2f} s :")
print(f"   • Neurones réels existants : {len(neurones):,}")
print(f"   • Synapses actives réelles : {len(liens):,}")

### 📚 3/6 : Téléchargement du Vrai Dataset Conversationnel Libre (CATIE-AQ NewsQuadFR)

In [ ]:
print("📚 3/6 : Téléchargement du Dataset Conversationnel (69 300 paires réelles)...")
url_dataset = "https://huggingface.co/datasets/CATIE-AQ/newsquadfr_fr_prompt_qa/resolve/main/train.parquet"
req = urllib.request.Request(url_dataset, headers={"User-Agent": "Mozilla/5.0"})

with urllib.request.urlopen(req) as resp:
    donnees_parquet = resp.read()

df_conversations = pd.read_parquet(io.BytesIO(donnees_parquet))
print(f"✅ {len(df_conversations):,} paires de Questions/Réponses réelles chargées en mémoire !")
print("Exemple d'échange 1 :", df_conversations['inputs'].iloc[0][:100], "➔", df_conversations['targets'].iloc[0])

### ⚡ 4/6 : Entraînement Épisodique RNI Transversal (Tissage Q ➔ R)

In [ ]:
print("⚡ 4/6 : Lancement du Tissage Synaptique Conversationnel...")
print("=" * 72)

# Index des connexions synaptiques existantes
arcs = {(s, d): w for s, d, w in cerveau["RNI_MATRICE"]["liens"]}
vocab = set(cerveau["RNI_MATRICE"]["neurones"])

nb_episodes = 10
nb_paires_total = len(df_conversations)
taille_lot = nb_paires_total // nb_episodes
t_start = time.time()
total_mots = 0
nouveaux_neurones = 0
nouveaux_liens = 0

for ep in range(1, nb_episodes + 1):
    t_ep = time.time()
    debut_idx = (ep - 1) * taille_lot
    fin_idx = min(nb_paires_total, ep * taille_lot)
    lot = df_conversations.iloc[debut_idx:fin_idx]
    mots_ep = 0

    for _, row in lot.iterrows():
        q_str = str(row['inputs'])
        r_str = str(row['targets'])

        q_tokens = [w.lower() for w in re.findall(r"\b\w+\b", q_str) if len(w) > 3]
        r_tokens = [w.lower() for w in re.findall(r"\b\w+\b", r_str) if len(w) > 3]

        mots_ep += len(q_tokens) + len(r_tokens)
        total_mots += len(q_tokens) + len(r_tokens)

        # Inscription des nouveaux neurones
        for w in q_tokens + r_tokens:
            cle_n = f"word_{w}"
            if cle_n not in vocab:
                vocab.add(cle_n)
                nouveaux_neurones += 1

        # 1. Tissage séquentiel intra-question
        for i in range(len(q_tokens) - 1):
            p = (f"word_{q_tokens[i]}", f"word_{q_tokens[i+1]}")
            arcs[p] = min(100, arcs.get(p, 10) + 2)

        # 2. Tissage TRANSVERSAL Q ➔ R (Le pont stimulus-réponse !)
        for q in q_tokens[:5]: # Concepts forts de la question
            for r in r_tokens[:3]: # Concepts pivots de la réponse
                p = (f"word_{q}", f"word_{r}")
                if p not in arcs:
                    arcs[p] = 20 # Premier ancrage transversal
                    nouveaux_liens += 1
                else:
                    arcs[p] = min(100, arcs[p] + 10) # Renforcement de dialogue

    dt_ep = time.time() - t_ep
    vitesse = mots_ep / max(0.001, dt_ep)
    energie_libre = 1.0 / (1.0 + len(arcs) * 0.00001)

    print(f"📍 [ÉPISODE {ep:02d}/{nb_episodes:02d}] ({debut_idx:,} ➔ {fin_idx:,} paires) en {dt_ep:.2f} s | Vitesse: {vitesse:,.0f} mots/s")
    print(f"   • Neurones Totaux: {len(vocab):,} (+{nouveaux_neurones}) | Synapses Totales: {len(arcs):,} (+{nouveaux_liens})")
    print(f"   • Énergie Libre: {energie_libre:.4f} | Corps ETH: Pouls {int(75 + min(25, vitesse/10000))} bpm (STABLE)")

duree_totale = time.time() - t_start
print("=" * 72)
print(f"🎉 ENTRAÎNEMENT CONVERSATIONNEL ACHEVÉ EN {duree_totale:.2f} s !")
print(f"   • Vitesse moyenne : {total_mots / max(0.01, duree_totale):,.0f} mots / seconde")

### 🧪 5/6 : Test de Résonance Conversationnelle Q ➔ R en Direct

In [ ]:
print("🧪 5/6 : Test d'Onde Synaptique sur les Nouveaux Arcs Conversationnels...")

# Index d'adjacence pour l'onde topologique
adj = {}
for (s, d), w in arcs.items():
    if s not in adj: adj[s] = []
    adj[s].append((d, w))

def sonder_question(question):
    mots = [w.lower() for w in re.findall(r"\b\w+\b", question) if len(w) > 3]
    print(f"\n👤 Question : « {question} »")
    reponses_allumees = {}
    for m in mots:
        cle = f"word_{m}"
        for cible, poids in adj.get(cle, []):
            if not cible.startswith(cle):
                reponses_allumees[cible] = reponses_allumees.get(cible, 0) + poids
    
    top = sorted(reponses_allumees.items(), key=lambda x: x[1], reverse=True)[:5]
    print("🤖 Neurones pivots allumés par résonance directe :")
    for n, score in top:
        nom = n[5:] if n.startswith("word_") else n
        print(f"   ⚡ [{nom}] (score d'activation = {score})")

sonder_question("Quelle manœuvre agressive reproche la France ?")
sonder_question("Qui est le fondateur de RATISS Labs à Yaoundé ?")
sonder_question("Explique-moi la fusion nucléaire")

### 💾 6/6 : Scellement Cryptographique SHA-256 et Export vers Google Drive

In [ ]:
print("💾 6/6 : Scellement et Exportation vers Google Drive...")

cerveau["RNI_MATRICE"]["neurones"] = sorted(list(vocab))
cerveau["RNI_MATRICE"]["liens"] = [[s, d, w] for (s, d), w in arcs.items()]
cerveau["META"]["DATE_DERNIERE_EDUCATION"] = time.strftime("%Y-%m-%d %H:%M:%S")
cerveau["META"]["DATASET_ENTRAINE"] = "CATIE-AQ/newsquadfr_fr_prompt_qa (69,300 paires)"
cerveau["META"]["TOTAL_MOTS_APPRIS"] = total_mots

# Scellement cryptographique SHA-256 inviolable
sans_sha = {k: v for k, v in cerveau.items() if k != "SIGNATURE_SHA256"}
signature = hashlib.sha256(json.dumps(sans_sha, sort_keys=True).encode("utf-8")).hexdigest()
cerveau["SIGNATURE_SHA256"] = signature

chemin_drive_sortie = "/content/drive/MyDrive/RATISS_CERVEAUX/RatissOne_Conversationnel.ratiss"
with gzip.open(chemin_drive_sortie, "wt", encoding="utf-8") as f:
    json.dump(cerveau, f)

taille_mo = os.path.getsize(chemin_drive_sortie) / (1024 * 1024)
print(f"\n🏆 SUCCÈS TOTAL !")
print(f"📁 Cerveau Conversationnel Scellé sauvegardé sur Google Drive :")
print(f"   ➔ {chemin_drive_sortie} ({taille_mo:.2f} Mo)")
print(f"🔒 Nouvelle Signature SHA-256 : {signature}")
print(f"👉 Prêt à être branché directement dans ton Docker ou ton runtime exclusif !")